# uTECH-GenUrban: Original Workflow

PopulationAgent → ActivityAgent → PlannerAgent → Gradient Boosting Regressor.

Configure the ignored `.env` and prepare your own inputs as described in `README.md` and `docs/INPUTS.md`. Running this notebook can call configured services and incur charges. No credentials, input data, or saved outputs are included.

This entry preserves the original file-level OD protocol. It is not the later controlled-hourly evaluation. See `docs/REPRODUCIBILITY.md` for the limited packaging and runtime fixes.


## 1. Setup and private configuration


In [ ]:
# Setup and private configuration
import os
import json
import re
import io
import time
from pathlib import Path
from typing import Dict, Any, List, Optional

from workflow_config import load_settings, safe_error, model_filename
settings = load_settings(validate_inputs=True)
API_KEY = settings.llm_configs[0]["api_key"]
API_BASE = settings.llm_configs[0]["base_url"]
config_list = settings.llm_configs

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import shape
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.manifold import MDS
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import autogen
from autogen import ConversableAgent, UserProxyAgent
from Utl import read_from_csv, acs_county_data, process_locs_buildings

os.chdir(settings.workspace)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.4)


## 2. Deterministic structural checks


In [ ]:
# Deterministic structural checks
import io
def hard_check_population(candidate: Dict[str, Any], State_fips, county_fips) -> Dict[str, Any]:
    violations = []
    if not isinstance(candidate, dict):
        return {"ok": False, "violations": ["candidate_not_dict"], "metrics": {}}

    if candidate.get("stage") != "population":
        violations.append("stage_must_be_population")

    clusters = candidate.get("clusters", None)
    if not isinstance(clusters, list) or len(clusters) == 0:
        violations.append("clusters_missing_or_empty")
        return {"ok": False, "violations": violations, "metrics": {"violations": len(violations)}}

    ids = []
    total = 0
    for i, c in enumerate(clusters):
        if not isinstance(c, dict):
            violations.append(f"cluster_{i}_not_dict")
            continue
        cid = c.get("cluster_id")
        if not cid or not isinstance(cid, str):
            violations.append(f"cluster_{i}_missing_cluster_id")
        else:
            ids.append(cid)

        pc = c.get("population_count")
        if not isinstance(pc, int) or pc <= 0:
            violations.append(f"cluster_{i}_population_count_invalid")
        else:
            total += pc

        if not c.get("demographic_summary"):
            violations.append(f"cluster_{i}_missing_demographic_summary")
        if not c.get("behavior_archetype"):
            violations.append(f"cluster_{i}_missing_behavior_archetype")

    if len(ids) != len(set(ids)):
        violations.append("cluster_id_not_unique")


    try:
        filename = f"ACS/{State_fips}_{county_fips}.csv"
        csv_content = read_from_csv(filename)
        df_acs = pd.read_csv(io.StringIO(csv_content))
        ACS_Total = df_acs['Estimate!!Total!!Total population'].iloc[0]
        

        tolerance = ACS_Total * 0.02
        
        if abs(total - ACS_Total) > tolerance:
            if total < ACS_Total:
                violations.append(f"total is smaller than ground truth by {abs(total-ACS_Total)}")
            if total > ACS_Total:
                violations.append(f"total is greater than ground truth by {abs(total-ACS_Total)}")
    except Exception as e:
        violations.append(f"Could not validate against ACS ground truth: {e}")

    metrics = {
        "total_population_count": total,
        "num_clusters": len(clusters),
        "violations": len(violations),
    }
    return {"ok": len(violations) == 0, "violations": violations, "metrics": metrics}


def hard_check_activity(candidate: Dict[str, Any], state_fips: str, county_fips: str, clusters: List[Dict[str, Any]] = None) -> Dict[str, Any]:
    tol = 1e-3
    violations = []

    if not isinstance(candidate, dict):
        return {"ok": False, "violations": ["candidate_not_dict"], "metrics": {}}

    if candidate.get("stage") != "activity":
        violations.append("stage_must_be_activity")

    ca = candidate.get("clusters_activity", None)
    if not isinstance(ca, list) or len(ca) == 0:
        violations.append("clusters_activity_missing_or_empty")
        return {"ok": False, "violations": violations, "metrics": {"violations": len(violations)}}


    if not clusters:
        violations.append("clusters_data_missing_from_kwargs")
        return {"ok": False, "violations": violations, "metrics": {"violations": len(violations)}}

    pop_ids = [c["cluster_id"] for c in clusters if isinstance(c, dict) and "cluster_id" in c]
    ca_ids = []
    bad_rows = 0

    required_keys = ["sleep", "work", "meal", "errand", "leisure"]

    for i, item in enumerate(ca):
        if not isinstance(item, dict):
            violations.append(f"clusters_activity_{i}_not_dict")
            continue
        cid = item.get("cluster_id")
        if not cid:
            violations.append(f"clusters_activity_{i}_missing_cluster_id")
            continue
        ca_ids.append(cid)

        hourly = item.get("hourly_activity", None)
        if not isinstance(hourly, list) or len(hourly) != 24:
            violations.append(f"{cid}_hourly_activity_must_have_24_entries")
            continue

        seen_hours = set()
        for hrow in hourly:
            if not isinstance(hrow, dict):
                violations.append(f"{cid}_hour_row_not_dict")
                bad_rows += 1
                continue

            hour = hrow.get("hour")
            if not isinstance(hour, int) or hour < 0 or hour > 23:
                violations.append(f"{cid}_invalid_hour_value")
                bad_rows += 1
                continue

            seen_hours.add(hour)
            s = 0.0
            for k in required_keys:
                v = hrow.get(k)
                if not isinstance(v, (int, float)):
                    violations.append(f"{cid}_hour_{hour}_{k}_not_number")
                    bad_rows += 1
                    continue
                if v < -tol or v > 1.0 + tol:
                    violations.append(f"{cid}_hour_{hour}_{k}_out_of_range")
                    bad_rows += 1
                s += float(v)

            if abs(s - 1.0) > 5e-2:
                violations.append(f"{cid}_hour_{hour}_shares_sum_not_1 (sum={s:.3f})")
                bad_rows += 1

        if seen_hours != set(range(24)):
            missing = sorted(list(set(range(24)) - seen_hours))
            violations.append(f"{cid}_missing_hours_{missing}")

    missing_clusters = sorted(list(set(pop_ids) - set(ca_ids)))
    extra_clusters = sorted(list(set(ca_ids) - set(pop_ids)))
    if missing_clusters:
        violations.append(f"missing_cluster_ids_in_activity: {missing_clusters}")
    if extra_clusters:
        violations.append(f"unknown_cluster_ids_in_activity: {extra_clusters}")

    metrics = {
        "num_population_clusters": len(pop_ids),
        "num_activity_clusters": len(ca_ids),
        "bad_rows": bad_rows,
        "violations": len(violations),
    }
    return {"ok": len(violations) == 0, "violations": violations, "metrics": metrics}


## 3. Population, activity, and verification agents


In [ ]:
# Population, activity, and verification agents
def build_llm_config(model_name: str) -> Dict[str, Any]:
    """
    Use the privately configured chat-completions service and model identifier.
    """
    return {
        "config_list": [
            {
                "model": model_name,
                "base_url": API_BASE,
                "api_key": API_KEY,
            }
        ],
        "temperature": 0.2,
    }


def build_agents(llm_config: Dict[str, Any]):
    user_proxy = UserProxyAgent(
        name="Runner",
        human_input_mode="NEVER",
        max_consecutive_auto_reply=0,
        default_auto_reply="TERMINATE",

        code_execution_config=False,
    )


    population_agent = ConversableAgent(
        name="PopulationGenerator",
        llm_config=llm_config,
        system_message=(
            "You are the Population Agent.\n"

            "Then return ONLY valid JSON with schema:\n"
            "{stage:'population', county_fips:str, state_fips:str, clusters:["
            "{cluster_id:str, population_count:int, demographic_summary:str, behavior_archetype:str}"
            "]}.\n"
            "demographic_summary str should be like Age (18-34: X%, 35-54: Y%, 55+: Z%), Income (Low: X%, Middle: Y%, High: Z%), Education (High School: X%, Some College: Y%, Bachelor's+: Z%), Gender (Male: X%, Female: Y%), Employment (Full-time: X%, Part-time: Y%, Unemployed: Z%), Travel Behavior (Public Transit: High, Car: Moderate)\n"
            "Make sure the generated clusters are corresponding to the ACS margins\n"
            "You must output ONLY valid JSON. No code blocks. No markdown. No extra text. The response must start with { and end with }"
            
        ),
    )

    activity_agent = ConversableAgent(
        name="ActivityGenerator",
        llm_config=llm_config,
        system_message=(
            "You are the Activity Agent.\n"
            "Input will include population clusters.\n"
            "Return ONLY valid JSON with schema:\n"
            "{stage:'activity', clusters_activity:["
            "{cluster_id:str, hourly_activity:["
            "{hour:int, sleep:float, work:float, meal:float, errand:float, leisure:float}"
            " x24]}"
            "]}.\n"
            "Per cluster per hour, shares must sum to 1.\n"
            "You must output ONLY valid JSON. No code blocks. No markdown. No extra text. The response must start with { and end with }"
        ),
    )

    pop_self_verifier = ConversableAgent(
        name="SelfVerifier",
        llm_config=llm_config,
        system_message=(
            "You are a verifier. You will receive candidate JSON and deterministic check_report.\n"

            "Output ONLY JSON:\n"
            "{should_patch:bool, verification_questions:[str], answers:[str], "
            "patch_plan:[{path:str, op:'replace'|'add'|'remove', value:any, reason:str}]}\n"
            "Use ONLY the candidate/check_report; do not invent external facts."
            "You must output ONLY valid JSON. No code blocks. No markdown. No extra text. The response must start with { and end with }"
        ),
    )

    act_self_verifier = ConversableAgent(
        name="SelfVerifier",
        llm_config=llm_config,
        system_message=(
            "You are a verifier. You will receive candidate JSON and deterministic check_report.\n"
            "Output ONLY JSON:\n"
            "{should_patch:bool, verification_questions:[str], answers:[str], "
            "patch_plan:[{path:str, op:'replace'|'add'|'remove', value:any, reason:str}]}\n"
            "Use ONLY the candidate/check_report; do not invent external facts."
            "You must output ONLY valid JSON. No code blocks. No markdown. No extra text. The response must start with { and end with }"
        ),
    )

    advocate = ConversableAgent(
        name="Advocate",
        llm_config=llm_config,
        system_message="Defend the candidate and address check_report concerns concisely.",
    )
    skeptic = ConversableAgent(
        name="Skeptic",
        llm_config=llm_config,
        system_message="Stress-test the candidate. Identify concrete inconsistencies or missing info.",
    )
    statistician = ConversableAgent(
        name="Statistician",
        llm_config=llm_config,
        system_message="Only discuss check_report metrics/violations and pass/fail.",
    )
    referee = ConversableAgent(
        name="Referee",
        llm_config=llm_config,
        system_message=(
            "You are the final decision maker.\n"
            "Primary rule: hard constraints dominate (check_report.ok must be true).\n"
            "Output ONLY JSON: {decision:'accept'|'reject', required_fixes:[str]}."
        ),
    )


    return user_proxy, population_agent, activity_agent, pop_self_verifier, act_self_verifier, advocate, skeptic, statistician, referee


## 4. Hybrid verification and generation prompts


In [ ]:
# Hybrid verification and generation prompts
import json
import re
from typing import Any, Dict, Optional
from autogen import ConversableAgent, UserProxyAgent

def extract_json_or_raise(text: str) -> dict:
    """Robustly extracts JSON from LLM text, handling markdown and filler."""
    if not text:
        raise ValueError("Empty response text.")
        

    clean_text = re.sub(r"^```(?:json|python)?\s*", "", text.strip(), flags=re.IGNORECASE)
    clean_text = re.sub(r"\s*```$", "", clean_text)
    

    start = clean_text.find("{")
    end = clean_text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON object found. Raw text: {text[:200]}...")
        
    return json.loads(clean_text[start:end+1])

def run_stage_hybrid(
    county: str,
    state: str,
    stage: str,
    generator_agent: ConversableAgent,
    prompt: str,
    user_proxy: UserProxyAgent,
    self_verifier: ConversableAgent,
    advocate: ConversableAgent,
    skeptic: ConversableAgent,
    statistician: ConversableAgent,
    referee: ConversableAgent,
    hard_check_fn,
    hard_check_kwargs: Optional[Dict[str, Any]] = None,
    max_rounds: int = 3,
    max_json_retries: int = 5,
) -> Dict[str, Any]:
    
    hard_check_kwargs = hard_check_kwargs or {}
    last_check = None
    last_candidate = None
    last_verdict = None

    def _chat_until_json(agent, message: str, clear: bool = True) -> Dict[str, Any]:

        user_proxy.initiate_chat(
            recipient=agent,
            message=message,
            clear_history=clear,
            summary_method="last_msg",
            silent=True
        )

        history = user_proxy.chat_messages.get(agent, [])
        last_txt = history[-1]["content"] if history else ""

        e = None
        try:
            return extract_json_or_raise(last_txt)
        except Exception as ex:
            e = ex


        for attempt in range(1, max_json_retries + 1):
            regen_msg = ("Return the previous answer as one valid JSON object only. "
                         "Keep the requested schema and intended values. "
                         "Do not include markdown, code fences, or explanatory text.")
            
            user_proxy.initiate_chat(
                recipient=agent,
                message=regen_msg,
                clear_history=False,
                silent=True
            )
            
            history = user_proxy.chat_messages.get(agent, [])
            last_txt = history[-1]["content"] if history else ""
            
            try:
                return extract_json_or_raise(last_txt)
            except Exception as ex2:
                e = ex2

        raise RuntimeError(f"[{stage}] Failed to obtain valid JSON from {agent.name}.")


    for r in range(1, max_rounds + 1):

        candidate = _chat_until_json(generator_agent, prompt, clear=True)


        check_report = hard_check_fn(candidate, state, county, **hard_check_kwargs)
        last_check, last_candidate = check_report, candidate


        vpacket = {"stage": stage, "candidate_json": candidate, "check_report": check_report, "county_fips":county, "state_fips":state}
        vjson = _chat_until_json(self_verifier, json.dumps(vpacket), clear=True)


        if vjson.get("should_patch") and vjson.get("patch_plan"):
            patch_req = {
                "stage": stage,
                "candidate_json": candidate,
                "check_report": check_report,
                "patch_plan": vjson["patch_plan"],
                "instruction": "Apply patch_plan and return revised JSON only.",
            }
            candidate = _chat_until_json(generator_agent, json.dumps(patch_req), clear=False)
            check_report = hard_check_fn(candidate, state, county, **hard_check_kwargs)
            last_check, last_candidate = check_report, candidate


        dpacket = {"stage": stage, "candidate_json": candidate, "check_report": check_report}
        dmsg = json.dumps(dpacket)


        user_proxy.initiate_chat(recipient=advocate, message=dmsg, clear_history=True)
        adv_history = user_proxy.chat_messages.get(advocate, [])
        adv = adv_history[-1]["content"] if adv_history else ""

        user_proxy.initiate_chat(recipient=skeptic, message=dmsg, clear_history=True)
        skp_history = user_proxy.chat_messages.get(skeptic, [])
        skp = skp_history[-1]["content"] if skp_history else ""

        user_proxy.initiate_chat(recipient=statistician, message=dmsg, clear_history=True)
        stat_history = user_proxy.chat_messages.get(statistician, [])
        stat = stat_history[-1]["content"] if stat_history else ""


        ref_packet = {
            "stage": stage,
            "candidate_json": candidate,
            "check_report": check_report,
            "debate": {"advocate": adv, "skeptic": skp, "statistician": stat},
        }
        verdict = _chat_until_json(referee, json.dumps(ref_packet), clear=True)
        last_verdict = verdict

        if verdict.get("decision") == "accept" and check_report.get("ok", False):
            return {
                "stage": stage,
                "round": r,
                "accepted": candidate,
                "check_report": check_report,
                "verdict": verdict,
            }

    return {
        "stage": stage,
        "round": max_rounds,
        "accepted": None,
        "check_report": last_check,
        "verdict": last_verdict or {"decision": "reject", "required_fixes": ["Max rounds reached."]},
        "last_candidate": last_candidate,
    }


def make_population_prompt(city: str, county_fips: str, state_fips: str) -> str:
    acs_text = acs_county_data(county_fips, state_fips)
    

    filename = f"ACS/{state_fips}_{county_fips}.csv"
    try:
        df_acs = pd.read_csv(filename)
        acs_total = int(df_acs['Estimate!!Total!!Total population'].iloc[0])
    except Exception:
        acs_total = "the total population in the ACS data"
        
    return json.dumps({

        "instruction": f"Generate population clusters. CRITICAL REQUIREMENT: The sum of 'population_count' across all your clusters MUST exactly equal {acs_total}.",
        "target_total_population": acs_total, 
        "city": city,
        "county_fips": county_fips,
        "state_fips": state_fips,
        "ACS_marginals": acs_text,
        "requirements": {
            "min_clusters": 4,
            "max_clusters": 8,
            "return_json_only": True,
            "cluster_fields": ["cluster_id", "population_count", "demographic_summary", "behavior_archetype"],
        }
    })

def make_activity_prompt(population_json: Dict[str, Any]) -> str:

    return json.dumps({
        "instruction": "Given population clusters, generate 24h aggregated activity shares per cluster.",
        "population_clusters": population_json["clusters"],
        "activity_types": ["sleep", "work", "meal", "errand", "leisure"],
        "requirements": {
            "hours": list(range(24)),
            "per_hour_sum_to_1": True,
            "return_json_only": True,
        }
    })


## 5. PlannerAgent mobility-force features


In [ ]:
# PlannerAgent mobility-force features
class MobilityPlanner:
    def __init__(self, model_cfg=None):

        llm_cfg = {"config_list": [model_cfg] if model_cfg else config_list}
        self.agent = autogen.AssistantAgent("Planner", llm_config=llm_cfg, system_message="Output JSON.")
        self.user = autogen.UserProxyAgent(
            "user", 
            human_input_mode="NEVER", 
            max_consecutive_auto_reply=0,
            code_execution_config=False
        )
        self.cache = {}

    def decide_forces(self, city_name, hour, day_type, schedules):
        cache_key = f"{city_name}_{day_type}_{hour}"
        if cache_key in self.cache: return self.cache[cache_key]
        

        summary = []
        for key, sched in schedules.items():
            if city_name in key:


                hourly_list = sched.get("hourly_activity", [])
                acts = next((item for item in hourly_list if item["hour"] == hour), {})
                
                if acts:

                    probs = {k: v for k, v in acts.items() if k != 'hour'}
                    top = max(probs, key=probs.get)
                    summary.append(f"{key}: {top} ({acts[top]:.2f})")
        context_str = "; ".join(summary)

        print(f"    [PLANNER] Reasoning for {city_name} @ {hour}:00...")
        
        prompt = f"""
        City: {city_name}, {day_type}, Hour {hour}.
        State: {context_str}.
        Task: 
        1. Analyze cultural/temporal drivers.
        2. Output 3 'Mobility Forces' (0.0-1.0).
        Output JSON: {{"reasoning": "...", "forces": {{"commute": <float>, "leisure": <float>, "home": <float>}}}}
        """
        try:
            res = self.user.initiate_chat(self.agent, message=prompt)
            data = json.loads(res.chat_history[-1]['content'].replace("```json","").replace("```","").strip())
            

            print(f"       Logic: {data['reasoning']}") 
            
            self.cache[cache_key] = data['forces']
            return data['forces']
        except:
            return {"commute": 0.3, "leisure": 0.3, "home": 0.3}


## 6. Spatial features and OD-flow regressor


In [ ]:
# Spatial features and OD-flow regressor
class HardDataEngine:
    def __init__(self, city_name, sociologist, planner):
        self.city_name = city_name
        self.soc = sociologist
        self.plan = planner
        self.coords = {}
        self.zones = []
        self.building_data = {}
        self.building_fallback_used = False
        self.model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
        self.scaler = StandardScaler()

    def ingest_buildings_dynamic(self, bbox, geojson_path):
        """Fetches Real Buildings with Local Caching."""
        import os
        print(f"\n [ENGINE] Fetching Overture Data for {self.city_name}...")
        try:

            gdf_shapes = gpd.read_file(geojson_path)
            if gdf_shapes.crs != "EPSG:4326":
                gdf_shapes = gdf_shapes.to_crs("EPSG:4326")
            gdf_shapes["shape_id"] = list(range(len(gdf_shapes)))
            num_regions = len(gdf_shapes)


            safe_city = self.city_name.replace(" ", "_")
            local_cache = f"Overture/{safe_city}_buildings.geojson"
            
            if os.path.exists(local_cache):
                print(f"    Found local Overture data! Loading from {local_cache}...")
                gdf_buildings = gpd.read_file(local_cache)
            else:
                print(f"    No local cache. Fetching from API...")
                locs = {self.city_name: bbox} 
                gdf_buildings = process_locs_buildings(locs)
                
                if not isinstance(gdf_buildings, gpd.GeoDataFrame):
                    gdf_buildings = gpd.GeoDataFrame(gdf_buildings, geometry='geometry', crs="EPSG:4326")
                elif gdf_buildings.crs is None:
                    gdf_buildings.set_crs("EPSG:4326", inplace=True)
                

                os.makedirs("Overture", exist_ok=True)
                gdf_buildings.to_file(local_cache, driver="GeoJSON")


            joined = gpd.sjoin(gdf_buildings, gdf_shapes, how="inner", predicate="within")


            self.building_data = {}
            agg = joined.groupby(["shape_id", "subtype"]).agg(total_capacity=("capacity", "sum")).reset_index()

            for _, row in agg.iterrows():
                sid = f"Region {int(row['shape_id'])}"
                if sid not in self.building_data: 
                    self.building_data[sid] = {'residential': 0, 'commercial': 0}
                
                subtype = str(row['subtype']).lower()
                cap = float(row['total_capacity'])
                
                if subtype in ['residential', 'apartments']:
                    self.building_data[sid]['residential'] += cap
                else:
                    self.building_data[sid]['commercial'] += cap
            
            print(f"    Spatially Joined buildings into {len(self.building_data)} zones.")

        except Exception as e:
            print(f"    Spatial Error: {safe_error(e)} (Using Fallback Density)")
            self.building_fallback_used = True

            try:
                num_regions = len(gpd.read_file(geojson_path))
            except:
                num_regions = 26
            self.building_data = {f"Region {i}": {'residential': 1000, 'commercial': 1000} for i in range(num_regions)}

            
    def learn_topology(self, files):
        print(f"    [TOPOLOGY] Learning Functional Map (MDS)...")
        flows = {}
        all_zones = set()
        for fp in files:
            try:
                df = pd.read_csv(fp)
                if 'flow_count' not in df.columns: df['flow_count'] = df.filter(like='Trips').sum(axis=1)
                for _, row in df.iterrows():
                    o, d, f = str(row['Origin']).strip(), str(row['Destination']).strip(), row['flow_count']
                    flows[(o,d)] = flows.get((o,d), 0) + f
                    all_zones.add(o); all_zones.add(d)
            except Exception as exc:
                print("Skipped unreadable OD input: " + safe_error(exc))
        self.zones = sorted(list(all_zones))
        if not self.zones: return False
        
        n = len(self.zones)
        dist_matrix = np.ones((n, n)) * 100.0
        max_flow = max(flows.values()) if flows else 1
        for (o, d), f in flows.items():
            if o in self.zones and d in self.zones:
                i, j = self.zones.index(o), self.zones.index(d)
                m = 1.0 - (f / max_flow)
                dist_matrix[i, j] = m; dist_matrix[j, i] = m
        np.fill_diagonal(dist_matrix, 0)
        
        mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42, normalized_stress=False)
        tx = mds.fit_transform(dist_matrix)
        for i, z in enumerate(self.zones): self.coords[z] = tx[i]
        return True

    def _prepare(self, files):
        X, y = [], []
        self.prepared_row_indices = []
        for fp in files:
            if not os.path.exists(fp): continue
            hour = 12
            match = re.search(r"(\d+)-(\d+)", fp)
            if match: hour = int(match.group(1))
            day_type = "Weekend" if "10-13" in fp else "Weekday"
            
            forces = self.plan.decide_forces(self.city_name, hour, day_type, self.soc.schedules)
            
            try:
                df = pd.read_csv(fp)
                if 'flow_count' not in df.columns: df['flow_count'] = df.filter(like='Trips').sum(axis=1)
                for row_index, row in df.iterrows():
                    o, d = str(row['Origin']).strip(), str(row['Destination']).strip()
                    if o not in self.coords or d not in self.coords or o==d: continue
                    

                    dist = np.linalg.norm(self.coords[o] - self.coords[d])

                    dest_comm = self.building_data.get(d, {}).get('commercial', 100) 
                    dest_res = self.building_data.get(d, {}).get('residential', 100)

                    feat = [dist, dest_comm, dest_res, hour, 1 if day_type=="Weekday" else 0,
                            forces['commute'], forces['leisure'], forces['home']]
                    
                    X.append(feat)
                    y.append(row['flow_count'])
                    self.prepared_row_indices.append(row_index)
            except Exception as exc:
                print("Skipped unreadable OD input: " + safe_error(exc))
        return np.array(X), np.array(y)

    def train_and_test(self, train_files, test_file):
        X_train, y_train = self._prepare(train_files)
        if len(X_train) == 0: return 0, [], []
        
        print(f"    [ML] Training on {len(X_train)} samples...")
        self.model.fit(self.scaler.fit_transform(X_train), y_train)
        
        X_test, y_true = self._prepare([test_file])
        self.test_row_indices = list(self.prepared_row_indices)
        if len(X_test) == 0: return 0, [], []
        y_pred = np.maximum(self.model.predict(self.scaler.transform(X_test)), 0)
        
        return r2_score(y_true, y_pred), y_true, y_pred


## 7. Optional workflow visualizations


In [ ]:
# Optional workflow visualizations
def plot_results(city_name, engine, r2, y_true, y_pred, sociologist):
    print(f"\nGenerating Plots for {city_name}...")
    

    activities = ['work', 'leisure']
    fig1, axes = plt.subplots(1, 2, figsize=(12, 4))
    hours = range(24)
    for i, act in enumerate(activities):
        ax = axes[i]
        for key, sched in sociologist.schedules.items():
            if city_name in key:
                cohort = key.split('_')[1]

                hourly_list = sched.get("hourly_activity", [])
                probs = []
                for h in hours:
                    acts = next((item for item in hourly_list if item["hour"] == h), {})
                    probs.append(acts.get(act, 0))
                ax.plot(hours, probs, label=cohort, linewidth=2)
        ax.set_title(f"{act.title()} Patterns")
        if i==0: ax.legend()
    plt.show()


    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    coords = np.array(list(engine.coords.values()))
    ax1.scatter(coords[:,0], coords[:,1], c='purple', s=100, alpha=0.6)
    for i, txt in enumerate(engine.coords.keys()):
        ax1.text(coords[i,0], coords[i,1], txt, fontsize=8)
    ax1.set_title(f"Latent Map (MDS)")
    
    ax2.scatter(y_true, y_pred, alpha=0.5, color='teal')
    ax2.plot([0, max(y_true)], [0, max(y_true)], 'r--')
    ax2.set_title(f"Accuracy (R²={r2:.3f})")
    ax2.set_xlabel("Actual"); ax2.set_ylabel("Predicted")
    plt.show()


    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    for ax, hour in [(ax1, 8), (ax2, 22)]:
        forces = {"commute":0.9,"leisure":0.1,"home":0.1} if hour==8 else {"commute":0.1,"leisure":0.9,"home":0.9}
        X_viz, pairs = [], []
        for o in engine.zones:
            for d in engine.zones:
                if o==d: continue
                dist = np.linalg.norm(engine.coords[o]-engine.coords[d])
                feat = [dist, 100, 100, hour, 1, forces['commute'], forces['leisure'], forces['home']]
                X_viz.append(feat)
                pairs.append((engine.coords[o], engine.coords[d]))
        flows = engine.model.predict(engine.scaler.transform(X_viz))
        max_f = max(flows)
        
        c = np.array(list(engine.coords.values()))
        ax.scatter(c[:,0], c[:,1], color='lightgrey', s=50)
        for i, (p1, p2) in enumerate(pairs):
            if flows[i] > max_f*0.2:
                ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color='blue' if hour==8 else 'red', alpha=flows[i]/max_f)
        ax.set_title(f"Hour {hour}: {'Commute' if hour==8 else 'Leisure'} Flow")
        ax.axis('off')
    plt.suptitle(f"Agent-Driven Flow Shift ({city_name})")
    plt.show()


## 8. Complete workflow execution


In [ ]:
# Complete workflow execution
cities_db = settings.cities


import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


os.makedirs("Outputs/Agent_JSONs", exist_ok=True)
os.makedirs("Outputs/Predicted_Flows", exist_ok=True)
os.makedirs("Outputs/Metrics", exist_ok=True)


def calculate_advanced_metrics(y_true, y_pred):
    y_pred = np.maximum(0, y_pred)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    

    sum_true, sum_pred = np.sum(y_true), np.sum(y_pred)
    cpc = 2 * np.sum(np.minimum(y_true, y_pred)) / (sum_true + sum_pred) if (sum_true + sum_pred) > 0 else 0
    
    return r2, rmse, mae, cpc

comparison_results = []

for city, data in cities_db.items():
    print(f"\n" + "="*60)
    print(f"  STARTING WORKFLOW: {city}")
    print("="*60)
    


    for cfg in config_list: 
        model_name = cfg['model']
        print(f"\n    MODEL: {model_name}...")
        safe_model_name = model_filename(model_name)
        
        try:

            llm_config = build_llm_config(model_name)
            llm_config["timeout"] = 300
            user_proxy, pop_gen, act_gen, pop_val, act_val, adv, skp, stat, ref = build_agents(llm_config)


            print("       Generating Population...")
            pop_prompt = make_population_prompt(city, data['county'], data['state'])
            

            pop_res = run_stage_hybrid(
                county=data['county'], 
                state=data['state'], 
                stage="population",
                generator_agent=pop_gen, 
                prompt=pop_prompt, 
                user_proxy=user_proxy,
                self_verifier=pop_val, 
                advocate=adv, 
                skeptic=skp, 
                statistician=stat, 
                referee=ref, 
                hard_check_fn=hard_check_population
            )
            population_data = pop_res["accepted"]
            if not population_data: raise ValueError("Population generation failed.")
            

            with open(f"Outputs/Agent_JSONs/{city}_{safe_model_name}_pop.json", "w") as f:
                json.dump(population_data, f, indent=4)


            print("       Generating Activity Schedules...")
            act_prompt = make_activity_prompt(population_data)
            

            act_res = run_stage_hybrid(
                county=data['county'], state=data['state'], stage="activity",
                generator_agent=act_gen, prompt=act_prompt, user_proxy=user_proxy,
                self_verifier=act_val, advocate=adv, skeptic=skp, statistician=stat, 
                referee=ref, hard_check_fn=hard_check_activity,
                hard_check_kwargs={"clusters": population_data["clusters"]}
            )
            activity_data = act_res["accepted"]
            if not activity_data: raise ValueError("Activity generation failed.")
            

            with open(f"Outputs/Agent_JSONs/{city}_{safe_model_name}_act.json", "w") as f:
                json.dump(activity_data, f, indent=4)


            print("       Running Physical Simulation & ML Engine...")
            plan = MobilityPlanner(cfg)
            
            class DummySociologist:
                def __init__(self):
                    self.schedules = {}
            soc = DummySociologist()
            for cluster_act in activity_data["clusters_activity"]:
                key = f"{city}_{cluster_act['cluster_id']}"
                soc.schedules[key] = cluster_act 

            eng = HardDataEngine(city, soc, plan)
            eng.ingest_buildings_dynamic(data['bbox'], data['geojson'])
            

            cluster_ids = [c['cluster_id'] for c in population_data["clusters"]]
            for i, (rid, rdata) in enumerate(eng.building_data.items()): 
                assigned_cluster = cluster_ids[i % len(cluster_ids)]
                rdata['cohort'] = f"{city}_{assigned_cluster}"
            

            eng.learn_topology(data['train'])
            raw_r2, true_flows, pred_flows = eng.train_and_test(data['train'], data['test'])
            

            r2, rmse, mae, cpc = calculate_advanced_metrics(true_flows, pred_flows)
            print(f"       {city} | R²: {r2:.4f} | CPC: {cpc:.4f} | RMSE: {rmse:.2f} | MAE: {mae:.2f}")
            

            

            if r2 > 0.0:
                test_df = pd.read_csv(data['test'])
                

                orig_col = 'Origin' if 'Origin' in test_df.columns else 'origin'
                dest_col = 'Destination' if 'Destination' in test_df.columns else 'destination'
                test_df = test_df.loc[eng.test_row_indices].copy()
                
                test_df['Actual_Flow'] = true_flows
                test_df['Predicted_Flow'] = np.maximum(0, pred_flows)
                

                def get_cluster(orig_id):
                    orig_str = f"Region {str(orig_id).replace('Region ', '').strip()}"
                    region_data = eng.building_data.get(orig_str, {})
                    return region_data.get('cohort', f'{city}_Unknown')
                
                test_df['Cluster'] = test_df[orig_col].apply(get_cluster)
                    
                test_df.to_csv(f"Outputs/Predicted_Flows/{city}_{safe_model_name}_OD_Flows.csv", index=False)
                
            if settings.show_plots and r2 > 0.8: plot_results(city, eng, r2, true_flows, pred_flows, soc)

            comparison_results.append({
                "City": city, 
                "Model": model_name, 
                "R2_Score": round(r2, 4),
                "CPC": round(cpc, 4),
                "RMSE": round(rmse, 2),
                "MAE": round(mae, 2),
                "Status": "complete",
                "Building_Fallback": eng.building_fallback_used
            })
            
        except Exception as e:
            print(f"       FAILED: {model_name} on {city} - Error: {safe_error(e)}")
            comparison_results.append({
                "City": city, "Model": model_name, 
                "R2_Score": np.nan, "CPC": np.nan, "RMSE": np.nan, "MAE": np.nan,
                "Status": "failed", "Error": safe_error(e)
            })


print("\n" + "="*60)
print(" COMPLETE WORKFLOW SUMMARY")
print("="*60)

df_results = pd.DataFrame(comparison_results)
if not df_results.empty:

    df_results.to_csv("Outputs/Metrics/Workflow_Metrics.csv", index=False)
    print(" Saved full metrics to Outputs/Metrics/Workflow_Metrics.csv\n")
    

    print("--- R² SCORES ---")
    print(df_results.pivot_table(index="City", columns="Model", values="R2_Score", aggfunc="max"))
    
    print("\n--- CPC (Common Part of Commuters) SCORES ---")
    print(df_results.pivot_table(index="City", columns="Model", values="CPC", aggfunc="max"))
